# DeepSeek-OCR PDF to Text Converter
Run this notebook in Google Colab to extract text from PDF files using DeepSeek-OCR

In [ ]:
# ===== PARAMETERS (Edit these) =====
INPUT_PDF = "input.pdf" # Path to your PDF file
OUTPUT_TXT = "output.txt" # Output text file path
START_PAGE = None # Start page (1-indexed), None for first page
END_PAGE = None # End page (1-indexed), None for last page
BASE_SIZE = 1024 # Image resolution (512/640/1024/1280)
# ===================================

In [ ]:
# Install dependencies
!pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers pillow pdf2image accelerate sentencepiece
!pip install -q flash-attn==2.7.3 --no-build-isolation

# Install poppler for PDF processing
!apt-get install -y poppler-utils

In [ ]:
# Import libraries
from transformers import AutoModel, AutoTokenizer
import torch
from pdf2image import convert_from_path
from PIL import Image
import os

In [ ]:
# Load model
print("Loading DeepSeek-OCR model...")
model_name = 'deepseek-ai/DeepSeek-OCR'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
 model_name, 
 _attn_implementation='flash_attention_2',
 trust_remote_code=True,
 use_safetensors=True,
 torch_dtype=torch.bfloat16
).eval().cuda()

print("Model loaded successfully!")

In [ ]:
# Convert PDF to images
print(f"Converting PDF pages to images...")
images = convert_from_path(
 INPUT_PDF,
 first_page=START_PAGE,
 last_page=END_PAGE
)

print(f"Processing {len(images)} pages...")

In [ ]:
# Process each page
all_text = []
prompt = "\n<|grounding|>Convert the document to markdown."

for i, img in enumerate(images, start=START_PAGE or 1):
 print(f"Processing page {i}...")
 
 # Save temporary image
 temp_img = f"temp_page_{i}.jpg"
 img.save(temp_img)
 
 # Run OCR
 result = model.infer(
 tokenizer,
 prompt=prompt,
 image_file=temp_img,
 output_path=None,
 base_size=BASE_SIZE,
 image_size=640,
 crop_mode=True,
 save_results=False,
 test_compress=True
 )
 
 all_text.append(f"--- Page {i} ---\n{result}\n")
 
 # Clean up temp file
 os.remove(temp_img)

# Save to output file
with open(OUTPUT_TXT, 'w', encoding='utf-8') as f:
 f.write('\n'.join(all_text))

print(f"\n✓ OCR complete! Output saved to: {OUTPUT_TXT}")
print(f"Total pages processed: {len(images)}")

# Display first 500 characters
print("\n--- Preview ---")
print(all_text[0][:500] + "..." if len(all_text[0]) > 500 else all_text[0])